Difference equations


In [1]:
import sympy as sp

k = sp.Symbol('k', integer=True)

T_n = sp.Function('T_n')
T_G = sp.Function('T_G')
t_hw = sp.Function('t_hw')
S = sp.Function('S')
Delta_up = sp.Function('Delta_up')

#  Define the State vector (x) and Input vector (u)
x_k = sp.Matrix([
    T_n(k), 
    t_hw(k), 
    S(k)
])

u_k = sp.Matrix([
    T_G(k), 
    t_hw(k),
    Delta_up(k)
])

# T_n[k] = T_G[k] + (Delta_up[k] + (T_G[k] - (T_G[k-1] + (t_hw[k] - t_hw[k-1]*S[k])))) / 2
T_n_expr = T_G(k) + (Delta_up(k) + (T_G(k) - (T_G(k-1) + (t_hw(k) - t_hw(k-1)*S(k))))) / 2

equation = sp.Eq(T_n(k), T_n_expr)

sp.pprint(equation)


        Δᵤₚ(k)   S(k)⋅t_hw(k - 1)   3⋅T_G(k)   T_G(k - 1)   t_hw(k)
Tₙ(k) = ────── + ──────────────── + ──────── - ────────── - ───────
          2             2              2           2           2


$$
x[k] = \begin{bmatrix}T_n[k] \\ t_{hw}[k] \\ S[k]\end{bmatrix},\quad u[k] = \begin{bmatrix} T_G[k]\\ t_{hw}[k] \\ \Delta up[k] \end{bmatrix}
$$


In [2]:
import sympy as sp

# 1. Define symbols for current [k] and past [k-1] time steps
T_n_k, t_hw_k, S_k = sp.symbols('T_n[k] t_hw[k] S[k]')
T_n_km1, t_hw_km1, S_km1 = sp.symbols('T_n[k-1] t_hw[k-1] S[k-1]')

T_G_k, t_hw_in_k, Delta_up_k = sp.symbols('T_G[k] t_hw_in[k] Delta_up[k]')
T_G_km1 = sp.symbols('T_G[k-1]')

# 2. Define State and Input Vectors
x_k = sp.Matrix([T_n_k, t_hw_k, S_k])
x_km1 = sp.Matrix([T_n_km1, t_hw_km1, S_km1])

# Using t_hw_in_k to distinguish the input t_hw from the state t_hw
u_k = sp.Matrix([T_G_k, t_hw_in_k, Delta_up_k])

# 3. Define the State Update Equations (f)
# Eq 1: Your expanded equation for T_n[k]
eq1 = T_G_k + (Delta_up_k + (T_G_k - (T_G_km1 + (t_hw_in_k - t_hw_km1 * S_k)))) / 2

# Eq 2 & 3: Placeholders! You will need to replace these with your actual system logic.
# For demonstration, let's assume t_hw state equals its input, and S is constant.
eq2 = t_hw_in_k 
eq3 = S_km1      

# Assemble the non-linear vector function: x[k] = f(x[k-1], u[k])
f = sp.Matrix([eq1, eq2, eq3])

# 4. Extract State-Space Matrices via Jacobian Linearization
# A Matrix: How past states (x_km1) affect current states (f)
A_matrix = f.jacobian(x_km1)

# B Matrix: How current inputs (u_k) affect current states (f)
B_matrix = f.jacobian(u_k)

# 5. Print the results
print("--- Non-Linear Equations f ---")
sp.pprint(f)
print("\n--- A Matrix (State Transition) ---")
sp.pprint(A_matrix)
print("\n--- B Matrix (Input) ---")
sp.pprint(B_matrix)


--- Non-Linear Equations f ---
⎡Δ_up[k]   S[k]⋅t_hw[k-1]   T_G[k-1]   3⋅T_G[k]   t_hw_in[k]⎤
⎢─────── + ────────────── - ──────── + ──────── - ──────────⎥
⎢   2            2             2          2           2     ⎥
⎢                                                           ⎥
⎢                        t_hw_in[k]                         ⎥
⎢                                                           ⎥
⎣                          S[k-1]                           ⎦
--- A Matrix (State Transition) ---
⎡   S[k]   ⎤
⎢0  ────  0⎥
⎢    2     ⎥
⎢          ⎥
⎢0   0    0⎥
⎢          ⎥
⎣0   0    1⎦
--- B Matrix (Input) ---
⎡3/2  -1/2  1/2⎤
⎢              ⎥
⎢ 0    1     0 ⎥
⎢              ⎥
⎣ 0    0     0 ⎦


In [3]:
import sympy as sp

# Define State Variables (k and k-1)
TN_k, TN_km1 = sp.symbols('T_N[k] T_N[k-1]')
tN_k, tN_km1 = sp.symbols('t_N[k] t_N[k-1]')
S_k, S_km1 = sp.symbols('S[k] S[k-1]')

# Define Inputs
TG_k = sp.symbols('T_G[k]')
thw_k = sp.symbols('t_{hw}[k]')
Dup_k = sp.symbols('Delta_up[k]')

# Define Constants
Kp = sp.symbols('K_p')

# --- 1. Equation for t_N[k] ---
eq_tN = thw_k

# --- 2. Equation for T_N[k] ---
# Elapsed hardware time
delta_thw = thw_k - tN_km1

# Predicted local elapsed time (thw_diff * skew_ratio)
delta_local = delta_thw * S_km1

# Predicted network time (my_stamp)
T_pred = TN_km1 + delta_local

# Downstream and Upstream delay calculations
delta_down = T_pred - TG_k
delay = (delta_down + Dup_k) / 2

# Final Network Time
eq_TN = sp.simplify(TG_k + delay)

# --- 3. Equation for S[k] ---
# gw_diff = current_true_time - old_gps
gw_diff = eq_TN - TN_km1

# err = gw_diff - predicted_elapsed
err = gw_diff - delta_local

# Final Skew Ratio
eq_S = sp.simplify(S_km1 + Kp * err)

sp.pprint("T_N[k] =", eq_TN)
sp.pprint("S[k] =", eq_S)


T_N[k] = Delta_up[k]/2 - S[k-1]*(t_N[k-1] - t_{hw}[k])/2 + T_G[k]/2 + T_N[k-1]/2
S[k] = K_p*(Delta_up[k] + S[k-1]*(t_N[k-1] - t_{hw}[k]) + T_G[k] - T_N[k-1])/2 + S[k-1]
